# KnowGen Multi-Agent System Test
## Testing Supervisor Agent & Retriever Agent with CNXHKH.docx

This notebook tests the two core agents on Vietnamese academic document (CNXHKH.docx)
- **Supervisor Agent**: Task classification, language detection, execution planning
- **Retriever Agent**: Query rewriting, document retrieval, ranking, evidence extraction

## Section 1: Setup Environment & Import Libraries

In [3]:
import sys
import os
import json
import logging
from pathlib import Path
from typing import Dict, List, Any
import pandas as pd
from dotenv import load_dotenv

# Setup paths
backend_dir = Path.cwd().parent.parent if "agents" in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(backend_dir))

# Load environment variables
load_dotenv()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Configuration
SOURCE_FILES = [
    r"H:\ĐH 1,2\NNHT.pdf",
    r"H:\ĐH 1,2\Vật lí 1\Chương 10 - Giao thoa ánh sáng.pdf",
    r"H:\ĐH 1,2\BT CNXHKH.docx",
]
VECTOR_STORE_DIR = backend_dir / "vector_store" / "faiss_index"

print(f"✓ Backend dir: {backend_dir}")
print("✓ Source files:")
for source in SOURCE_FILES:
    print(f"   - {source}")
print(f"✓ Vector store dir: {VECTOR_STORE_DIR}")
print(f"✓ Ready to test agents!")

✓ Backend dir: h:\KnowGen Project\backend
✓ Source files:
   - H:\ĐH 1,2\NNHT.pdf
   - H:\ĐH 1,2\Vật lí 1\Chương 10 - Giao thoa ánh sáng.pdf
   - H:\ĐH 1,2\BT CNXHKH.docx
✓ Vector store dir: h:\KnowGen Project\backend\vector_store\faiss_index
✓ Ready to test agents!


## Section 2: Install & Verify Dependencies

In [4]:
import subprocess
import sys

# Define required packages
required_packages = [
    'langgraph',
    'langchain',
    'langchain-core',
    'langchain-openai',
    'langchain-huggingface',
    'langchain-community',
    'faiss-cpu',
    'python-docx',
    'pandas',
    'python-dotenv'
]

print("=" * 80)
print("📦 CHECKING AND INSTALLING DEPENDENCIES")
print("=" * 80)

missing_packages = []

# Check which packages are missing
for package in required_packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package}")
    except ImportError:
        print(f"❌ {package} (MISSING)")
        missing_packages.append(package)

if missing_packages:
    print(f"\n⚠️  Installing {len(missing_packages)} missing package(s)...")
    for package in missing_packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
            print(f"✓ Installed {package}")
        except Exception as e:
            print(f"❌ Failed to install {package}: {e}")
else:
    print("\n✓ All required packages are already installed!")

print("\n" + "=" * 80)

📦 CHECKING AND INSTALLING DEPENDENCIES
✓ langgraph
✓ langchain
✓ langchain-core
✓ langchain-openai
✓ langchain-huggingface
✓ langchain-community
❌ faiss-cpu (MISSING)
❌ python-docx (MISSING)
✓ pandas
❌ python-dotenv (MISSING)

⚠️  Installing 3 missing package(s)...
✓ Installed faiss-cpu
✓ Installed python-docx
✓ Installed python-dotenv



In [5]:
print("=" * 80)
print("✅ VERIFYING DEPENDENCY INSTALLATION")
print("=" * 80)

# Test actual imports to verify installation worked
verification_imports = {
    'langgraph': 'from langgraph.graph import StateGraph, START, END',
    'langchain_core': 'from langchain_core.documents import Document',
    'langchain_huggingface': 'from langchain_huggingface import HuggingFaceEmbeddings',
    'langchain_community': 'from langchain_community.vectorstores import FAISS',
    'docx': 'from docx import Document as DocxDocument',
    'pandas': 'import pandas as pd',
    'dotenv': 'from dotenv import load_dotenv'
}

success_count = 0
failed_imports = []

for package_name, import_statement in verification_imports.items():
    try:
        exec(import_statement)
        print(f"✓ {package_name:30} - OK")
        success_count += 1
    except Exception as e:
        print(f"❌ {package_name:30} - FAILED: {str(e)[:40]}")
        failed_imports.append(package_name)

print("\n" + "-" * 80)
if failed_imports:
    print(f"⚠️  {len(failed_imports)}/{len(verification_imports)} imports failed")
    print(f"Failed packages: {', '.join(failed_imports)}")
    print("\nAttempting to reinstall failed packages...")
    for pkg in failed_imports:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            print(f"✓ Reinstalled {pkg}")
        except:
            print(f"❌ Could not reinstall {pkg}")
else:
    print(f"✓ All {len(verification_imports)} dependencies verified successfully!")

print("=" * 80)

✅ VERIFYING DEPENDENCY INSTALLATION
✓ langgraph                      - OK
✓ langchain_core                 - OK
✓ langchain_huggingface          - OK
✓ langchain_community            - OK
✓ docx                           - OK
✓ pandas                         - OK
✓ dotenv                         - OK

--------------------------------------------------------------------------------
✓ All 7 dependencies verified successfully!


## Section 3: Load & Configure Agents

In [6]:
print("=" * 80)
print("📦 LOADING AGENT MODULES")
print("=" * 80)

# Step 1: Import AgentState first from dedicated module (no circular deps)
try:
    from app.agents.agent_state import AgentState
    print("✓ AgentState imported from agent_state")
except Exception as e:
    logger.error(f"Failed to load AgentState: {e}")
    print(f"❌ Error loading AgentState: {e}")
    raise

# Step 2: Import supervisor_node
try:
    from app.agents.supervisor_agent import supervisor_node
    print("✓ Supervisor Agent imported")
except Exception as e:
    logger.error(f"Failed to load supervisor_node: {e}")
    print(f"❌ Error loading supervisor_node: {e}")
    raise

# Step 3: Import LLMClient  
try:
    from app.llm.llm_client import LLMClient
    print("✓ LLMClient imported")
except Exception as e:
    logger.error(f"Failed to load LLMClient: {e}")
    print(f"❌ Error loading LLMClient: {e}")
    raise

# Step 4: Import RetrieverAgent (now after AgentState is available)
try:
    from app.agents.retriever_agent import RetrieverAgent
    print("✓ Retriever Agent imported")
except Exception as e:
    logger.error(f"Failed to load RetrieverAgent: {e}")
    print(f"❌ Error loading RetrieverAgent: {e}")
    raise

# Step 5: Import remaining workflow components (optional)
try:
    from app.agents.multi_agent import build_knowgen_workflow, run_knowgen_pipeline
    print("✓ Multi-Agent framework imported")
except ImportError as e:
    logger.warning(f"Workflow framework not available: {e}")
    print(f"⚠️  Workflow framework note: Complete framework not yet available (optional)")
except Exception as e:
    logger.error(f"Failed to load workflow: {e}")
    print(f"❌ Error loading workflow: {e}")

# Step 6: Initialize LLM client
try:
    llm_client = LLMClient()
    logger.info("✓ LLMClient initialized")
    print("✓ LLMClient initialized successfully")
except Exception as e:
    logger.error(f"Failed to initialize LLMClient: {e}")
    print(f"⚠️  Warning: LLMClient initialization issue - {e}")
    print("   → Check your .env file has valid API keys (OPENAI_API_KEY, etc.)")

print("\n" + "=" * 80)
print("✓ All required agents loaded and ready for testing!")
print("=" * 80)

📦 LOADING AGENT MODULES
✓ AgentState imported from agent_state


2026-04-15 13:36:42,340 - knowgen.retriever - INFO - Loading embedding model: intfloat/multilingual-e5-base


✓ Supervisor Agent imported
✓ LLMClient imported


2026-04-15 13:36:45,194 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2026-04-15 13:36:45,194 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: intfloat/multilingual-e5-base
2026-04-15 13:36:54,878 - knowgen.retriever - INFO - Loading FAISS index from: h:\KnowGen Project\backend\app\agents\vector_store\faiss_index
2026-04-15 13:36:54,878 - faiss.loader - INFO - Loading faiss with AVX2 support.
2026-04-15 13:36:54,904 - faiss.loader - INFO - Successfully loaded faiss with AVX2 support.
2026-04-15 13:36:54,912 - knowgen.retriever - INFO - FAISS index loaded successfully
2026-04-15 13:36:54,934 - __main__ - INFO - ✓ LLMClient initialized


✓ Retriever Agent imported
✓ Multi-Agent framework imported
✓ LLMClient initialized successfully

✓ All required agents loaded and ready for testing!


## Section 4: Run ETL for the 3 Source Files

In [7]:
import time
from app.ingestion.extract import extract_pdf_to_md, extract_docx_to_md
from app.ingestion.transform import transform_documents
from app.ingestion.load import load_documents_to_faiss

print("=" * 80)
print("📥 RUNNING ETL FOR 3 INPUT FILES")
print("=" * 80)

extracted_documents = []
extract_errors = []

start_etl = time.perf_counter()
for index, source_path in enumerate(SOURCE_FILES, 1):
    print(f"\n[{index}/{len(SOURCE_FILES)}] Processing: {source_path}")
    if not os.path.exists(source_path):
        print(f"❌ Missing file: {source_path}")
        extract_errors.append({"path": source_path, "error": "File not found"})
        continue

    try:
        if source_path.lower().endswith(".pdf"):
            document = extract_pdf_to_md(source_path)
        elif source_path.lower().endswith(".docx"):
            document = extract_docx_to_md(source_path)
        else:
            raise ValueError("Unsupported file format")

        length_chars = len(document.content)
        length_words = len(document.content.split())
        print(f"✓ Extracted: {document.id} ({document.source_type})")
        print(f"   characters={length_chars:,} | words={length_words:,}")
        extracted_documents.append(document)

    except Exception as e:
        print(f"❌ Extraction failed: {e}")
        extract_errors.append({"path": source_path, "error": str(e)})

print("\n" + "=" * 80)
print("🔧 TRANSFORMING EXTRACTED DOCUMENTS")
print("=" * 80)

if extracted_documents:
    start_transform = time.perf_counter()
    transformed_documents = transform_documents(extracted_documents)
    elapsed_transform = time.perf_counter() - start_transform

    print(f"\n✓ Transformed {len(transformed_documents)} document(s) in {elapsed_transform:.1f}s")
    for transformed in transformed_documents:
        print(f"   - {transformed.doc_id}: {len(transformed.chunks)} chunks")
else:
    transformed_documents = []
    print("⚠️  No documents available for transform.")

print("\n" + "=" * 80)
print("🧠 LOADING DOCUMENTS INTO FAISS VECTOR STORE")
print("=" * 80)

if transformed_documents:
    VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)
    faiss_store = load_documents_to_faiss(
        transformed_documents,
        save_dir=str(VECTOR_STORE_DIR)
    )

    if faiss_store:
        print(f"✓ FAISS index created and saved at: {VECTOR_STORE_DIR}")
    else:
        print("❌ FAISS index creation failed. Check messages above.")
else:
    faiss_store = None
    print("⚠️  Skipped FAISS loading because no transformed documents were generated.")

elapsed_total = time.perf_counter() - start_etl
print("\n" + "=" * 80)
print(f"✅ ETL complete in {elapsed_total:.1f}s")
print(f"   documents extracted: {len(extracted_documents)}")
print(f"   documents transformed: {len(transformed_documents)}")
print(f"   vector store ready: {faiss_store is not None}")
print("=" * 80)

📥 RUNNING ETL FOR 3 INPUT FILES

[1/3] Processing: H:\ĐH 1,2\NNHT.pdf
Extracting PDF (Markdown): H:\ĐH 1,2\NNHT.pdf
✓ Extracted: NNHT.pdf (pdf)
   characters=92,078 | words=16,977

[2/3] Processing: H:\ĐH 1,2\Vật lí 1\Chương 10 - Giao thoa ánh sáng.pdf
Extracting PDF (Markdown): H:\ĐH 1,2\Vật lí 1\Chương 10 - Giao thoa ánh sáng.pdf
✓ Extracted: Chương 10 - Giao thoa ánh sáng.pdf (pdf)
   characters=15,810 | words=2,994

[3/3] Processing: H:\ĐH 1,2\BT CNXHKH.docx
Extracting DOCX (Markdown): H:\ĐH 1,2\BT CNXHKH.docx


2026-04-15 13:37:58,543 - app.ingestion.transform - INFO - Transforming document: NNHT.pdf
2026-04-15 13:37:58,660 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


✓ Extracted: BT CNXHKH.docx (docx)
   characters=8,756 | words=1,924

🔧 TRANSFORMING EXTRACTED DOCUMENTS


2026-04-15 13:38:16,481 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-15 13:38:16,569 - app.ingestion.transform - INFO - Done 'NNHT.pdf': summary=ok, chunks=356
2026-04-15 13:38:16,571 - app.ingestion.transform - INFO - Transforming document: Chương 10 - Giao thoa ánh sáng.pdf
2026-04-15 13:38:16,582 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.
2026-04-15 13:38:24,061 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-15 13:38:24,092 - app.ingestion.transform - INFO - Done 'Chương 10 - Giao thoa ánh sáng.pdf': summary=ok, chunks=18
2026-04-15 13:38:24,094 - app.ingestion.transform - INFO - Transforming document: BT CNXHKH.docx
2026-04-15 13:38:24,103 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.
2026-04-15 13:38:34,905 - httpx - I


✓ Transformed 3 document(s) in 36.4s
   - NNHT.pdf: 356 chunks
   - Chương 10 - Giao thoa ánh sáng.pdf: 18 chunks
   - BT CNXHKH.docx: 25 chunks

🧠 LOADING DOCUMENTS INTO FAISS VECTOR STORE


2026-04-15 13:38:41,621 - app.ingestion.load - INFO - Flattened 3 TransformedDocument(s) into 399 Langchain Document(s) (added 'passage: ' prefix)
2026-04-15 13:38:41,637 - app.ingestion.load - INFO - Creating Vector Embeddings for 399 chunks and indexing into FAISS... (This process may take time depending on your machine)
2026-04-15 13:40:52,692 - app.ingestion.load - INFO - Successfully created FAISS index in RAM!
2026-04-15 13:40:52,697 - app.ingestion.load - INFO - Successfully saved FAISS Index database to directory: h:\KnowGen Project\backend\vector_store\faiss_index


✓ FAISS index created and saved at: h:\KnowGen Project\backend\vector_store\faiss_index

✅ ETL complete in 230.1s
   documents extracted: 3
   documents transformed: 3
   vector store ready: True


## Section 5: Test Supervisor Agent

In [8]:
# Test queries for the three documents
# These queries include both factual QA and quiz-style prompts.
test_queries = [
    "Cho biết các đặc điểm chính của chủ nghĩa xã hội khoa học.",
    "Tóm tắt nội dung chính của tài liệu NNHT.pdf về nguyên lý học tập.",
    "Định lý Pythagore phát biểu như thế nào và có ứng dụng gì trong hình học?",
    "Tạo một câu hỏi trắc nghiệm về giao thoa ánh sáng và các điều kiện cần thiết."
]

# Store supervisor results
supervisor_results = {}

print("=" * 80)
print("🤖 TESTING SUPERVISOR AGENT")
print("=" * 80)

for i, query in enumerate(test_queries, 1):
    print(f"\n[Query {i}/{len(test_queries)}] {query}")
    print("-" * 80)
    
    try:
        # Create initial state
        initial_state = {"user_query": query}
        
        # Run supervisor node
        supervisor_output = supervisor_node(initial_state)
        
        # Store result
        supervisor_results[f"query_{i}"] = supervisor_output
        
        # Display results
        if supervisor_output.get('task_type'):
            print(f"✓ Task Type: {supervisor_output.get('task_type')}")
            print(f"✓ Language: {supervisor_output.get('language')}")
            print(f"✓ Intent: {supervisor_output.get('intent_summary', 'N/A')[:70]}")
            
            plan = supervisor_output.get('plan', {})
            context = plan.get('context_needed', 'N/A')
            steps = plan.get('steps', [])
            
            print(f"✓ Context Needed: {context[:70]}")
            if steps:
                print(f"✓ Steps ({len(steps)}):")
                for step_idx, step in enumerate(steps[:3], 1):
                    print(f"   {step_idx}. {step[:60]}")
        else:
            print(f"⚠️  Missing task_type in response")
            print(f"   Raw output: {supervisor_output}")
        
    except Exception as e:
        print(f"❌ Error processing query: {e}")
        import traceback
        traceback.print_exc()
        supervisor_results[f"query_{i}"] = {"error": str(e)}

print("\n" + "=" * 80)
print(f"✓ Supervisor Agent testing complete!")
print(f"   Successfully processed: {len([r for r in supervisor_results.values() if 'error' not in r])}/{len(test_queries)} queries")
print("=" * 80)

2026-04-15 13:40:56,043 - knowgen.supervisor - INFO - === SUPERVISOR AGENT RUNNING ===
2026-04-15 13:40:56,055 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


🤖 TESTING SUPERVISOR AGENT

[Query 1/4] Cho biết các đặc điểm chính của chủ nghĩa xã hội khoa học.
--------------------------------------------------------------------------------


2026-04-15 13:41:04,674 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-15 13:41:04,689 - knowgen.supervisor - INFO - === SUPERVISOR AGENT RUNNING ===
2026-04-15 13:41:04,689 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


✓ Task Type: qa
✓ Language: vi
✓ Intent: Người dùng muốn biết các đặc điểm chính của chủ nghĩa xã hội khoa học.
✓ Context Needed: chủ nghĩa xã hội khoa học, đặc điểm, tính chất
✓ Steps (3):
   1. Bước 1: Sử dụng tác nhân truy xuất để tìm kiếm các tài liệu 
   2. Bước 2: Sử dụng tác nhân QA để tổng hợp và trích xuất các đặ
   3. Bước 3: Trình bày câu trả lời bằng tiếng Việt.

[Query 2/4] Tóm tắt nội dung chính của tài liệu NNHT.pdf về nguyên lý học tập.
--------------------------------------------------------------------------------


2026-04-15 13:41:07,807 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-15 13:41:07,820 - knowgen.supervisor - INFO - === SUPERVISOR AGENT RUNNING ===
2026-04-15 13:41:07,820 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


✓ Task Type: qa
✓ Language: vi
✓ Intent: Người dùng muốn tóm tắt các nguyên lý học tập được đề cập trong tài li
✓ Context Needed: NNHT.pdf, nguyên lý học tập
✓ Steps (4):
   1. Step 1: Truy xuất tài liệu 'NNHT.pdf' từ kho tri thức.
   2. Step 2: Phân tích nội dung của 'NNHT.pdf' để xác định và trí
   3. Step 3: Tóm tắt các nguyên lý học tập đã được trích xuất một

[Query 3/4] Định lý Pythagore phát biểu như thế nào và có ứng dụng gì trong hình học?
--------------------------------------------------------------------------------


2026-04-15 13:41:10,470 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-15 13:41:10,476 - knowgen.supervisor - INFO - === SUPERVISOR AGENT RUNNING ===
2026-04-15 13:41:10,476 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


✓ Task Type: qa
✓ Language: vi
✓ Intent: Người dùng muốn biết định lý Pythagore phát biểu như thế nào và các ứn
✓ Context Needed: Định lý Pythagore, phát biểu định lý Pythagore, ứng dụng định lý Pytha
✓ Steps (2):
   1. Bước 1: Sử dụng bộ truy xuất để tìm các tài liệu liên quan đ
   2. Bước 2: Sử dụng tác nhân QA để tổng hợp thông tin đã truy xu

[Query 4/4] Tạo một câu hỏi trắc nghiệm về giao thoa ánh sáng và các điều kiện cần thiết.
--------------------------------------------------------------------------------


2026-04-15 13:41:13,171 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


✓ Task Type: quiz
✓ Language: vi
✓ Intent: Người dùng muốn tạo một câu hỏi trắc nghiệm về giao thoa ánh sáng và c
✓ Context Needed: giao thoa ánh sáng, điều kiện cần thiết cho giao thoa ánh sáng
✓ Steps (3):
   1. Bước 1: Sử dụng tác nhân truy xuất để tìm kiếm thông tin liê
   2. Bước 2: Dựa trên thông tin thu được, tác nhân tạo câu hỏi tr
   3. Bước 3: Trình bày câu hỏi trắc nghiệm đã tạo cho người dùng.

✓ Supervisor Agent testing complete!
   Successfully processed: 4/4 queries


## Section 6: Test Retriever Agent

In [9]:
# 🔄 CRITICAL: Clear module cache AND re-import fresh versions
# This ensures we load the UPDATED retriever_agent.py with fixed method names

import sys
import importlib

print("=" * 80)
print("🔄 CLEARING MODULE CACHE & RELOADING MODULES")
print("=" * 80)

# List of modules to reload
modules_to_clear = [
    'app.agents.retriever_agent',
    'app.llm.llm_client',
    'app.agents.agent_state'
]

print("\n1️⃣  Removing from sys.modules cache:")
print("-" * 80)
for module_name in modules_to_clear:
    if module_name in sys.modules:
        del sys.modules[module_name]
        print(f"✓ Cleared {module_name}")
    else:
        print(f"⊘ {module_name} not in cache")

print("\n2️⃣  Re-importing fresh modules:")
print("-" * 80)

# Re-import to load fresh versions
try:
    from app.agents.agent_state import AgentState
    print("✓ Re-imported AgentState (fresh)")
except Exception as e:
    print(f"❌ Error re-importing AgentState: {e}")

try:
    from app.llm.llm_client import LLMClient
    print("✓ Re-imported LLMClient (fresh)")
except Exception as e:
    print(f"❌ Error re-importing LLMClient: {e}")

try:
    from app.agents.retriever_agent import RetrieverAgent
    print("✓ Re-imported RetrieverAgent (fresh) ← THIS NOW HAS similarity_search_with_relevance_scores()")
except Exception as e:
    print(f"❌ Error re-importing RetrieverAgent: {e}")

print("\n✓ All modules reloaded! Ready to test with updated code.")
print("=" * 80)

2026-04-15 13:41:24,856 - knowgen.retriever - INFO - Loading embedding model: intfloat/multilingual-e5-base
2026-04-15 13:41:24,863 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2026-04-15 13:41:24,864 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: intfloat/multilingual-e5-base


🔄 CLEARING MODULE CACHE & RELOADING MODULES

1️⃣  Removing from sys.modules cache:
--------------------------------------------------------------------------------
✓ Cleared app.agents.retriever_agent
✓ Cleared app.llm.llm_client
✓ Cleared app.agents.agent_state

2️⃣  Re-importing fresh modules:
--------------------------------------------------------------------------------
✓ Re-imported AgentState (fresh)
✓ Re-imported LLMClient (fresh)


2026-04-15 13:41:32,384 - knowgen.retriever - INFO - Loading FAISS index from: h:\KnowGen Project\backend\app\agents\vector_store\faiss_index
2026-04-15 13:41:32,384 - knowgen.retriever - INFO - FAISS index loaded successfully


✓ Re-imported RetrieverAgent (fresh) ← THIS NOW HAS similarity_search_with_relevance_scores()

✓ All modules reloaded! Ready to test with updated code.


In [10]:
# Check if FAISS index exists
print("=" * 80)
print("🔍 TESTING RETRIEVER AGENT")
print("=" * 80)

retriever = None
retriever_results = {}

if not os.path.exists(VECTOR_STORE_DIR):
    print(f"\n⚠️  FAISS index not found at {VECTOR_STORE_DIR}")
    print("\n📋 HOW TO CREATE VECTOR EMBEDDINGS:")
    print("-" * 80)
    print("1. Run ETL pipeline to process the three source files:")
    print("   python scripts/run_etl.py")
    print("2. This will:")
    print("   - Extract text from NNHT.pdf, Giao thoa ánh sáng.pdf, and BT CNXHKH.docx")
    print("   - Transform and chunk the content")
    print("   - Create FAISS vector embeddings")
    print("   - Save index to:", VECTOR_STORE_DIR)
    print("\n3. After ETL completes, re-run this cell")
    print("-" * 80)
    print("\n✓ Skipping Retriever Agent tests (waiting for FAISS index)")
else:
    # Initialize retriever
    try:
        retriever = RetrieverAgent(vector_store_dir=str(VECTOR_STORE_DIR))
        print("\n✓ Retriever Agent initialized successfully")
        
        # Test retriever with supervisor outputs
        for query_idx, (query_key, supervisor_output) in enumerate(supervisor_results.items(), 1):
            if "error" in supervisor_output:
                print(f"\n⚠️  Skipping Query {query_idx} due to supervisor error")
                continue
            
            print(f"\n{'='*80}")
            print(f"[Query {query_idx}] {test_queries[query_idx-1]}")
            print(f"{'='*80}")
            
            try:
                # Create AgentState from supervisor output
                agent_state = AgentState(
                    user_query=test_queries[query_idx-1],
                    task_type=supervisor_output.get('task_type', 'qa'),
                    language=supervisor_output.get('language', 'en'),
                    intent_summary=supervisor_output.get('intent_summary', ''),
                    plan=supervisor_output.get('plan', {})
                )
                
                # Run retriever
                retriever_output = retriever.run(agent_state)
                retriever_results[query_key] = retriever_output
                
                # Display results
                print(f"✓ Rewritten Query: {retriever_output.get('rewritten_query', 'N/A')}")
                print(f"✓ Documents Retrieved: {len(retriever_output.get('retrieved_docs', []))}")
                
                # Show top documents
                retrieved_docs = retriever_output.get('retrieved_docs', [])
                if retrieved_docs:
                    print(f"\n📄 Top Documents:")
                    for doc_idx, doc in enumerate(retrieved_docs[:3], 1):
                        score = doc.metadata.get('similarity_score', 0) if doc.metadata else 0
                        header = doc.metadata.get('header_path', 'N/A') if doc.metadata else 'N/A'
                        print(f"  [{doc_idx}] Score: {score:.3f} | Header: {header}")
                        print(f"      Preview: {doc.page_content[:80].strip()}...")
                    
                    # Show evidence
                    evidence = retriever_output.get('evidence_summary', [])
                    if evidence:
                        print(f"\n💡 Evidence Summaries ({len(evidence)} items):")
                        for ev_idx, ev in enumerate(evidence[:3], 1):
                            print(f"  {ev_idx}. {ev[:80]}")
                else:
                    print("⚠️  No documents retrieved for this query")
                
            except Exception as e:
                logger.error(f"Retriever error for query {query_idx}: {e}")
                print(f"❌ Error processing retriever: {e}")
                import traceback
                traceback.print_exc()
                retriever_results[query_key] = {"error": str(e)}
        
    except Exception as e:
        logger.error(f"Failed to initialize retriever: {e}")
        print(f"\n❌ Failed to initialize Retriever Agent: {e}")
        print(f"   Error details: {str(e)}")
        import traceback
        traceback.print_exc()

print("\n" + "=" * 80)
if retriever:
    print(f"✓ Retriever Agent testing complete! ({len(retriever_results)} results)")
else:
    print(f"⚠️  Retriever Agent testing skipped (FAISS index not available)")
print("=" * 80)

2026-04-15 13:41:34,935 - knowgen.retriever - INFO - Loading embedding model: intfloat/multilingual-e5-base
2026-04-15 13:41:34,938 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2026-04-15 13:41:34,939 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: intfloat/multilingual-e5-base


🔍 TESTING RETRIEVER AGENT


2026-04-15 13:41:41,554 - knowgen.retriever - INFO - Loading FAISS index from: h:\KnowGen Project\backend\vector_store\faiss_index
2026-04-15 13:41:41,570 - knowgen.retriever - INFO - FAISS index loaded successfully
2026-04-15 13:41:41,595 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.



✓ Retriever Agent initialized successfully

[Query 1] Cho biết các đặc điểm chính của chủ nghĩa xã hội khoa học.


2026-04-15 13:41:48,345 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-15 13:41:48,356 - knowgen.retriever - INFO - Query rewritten: 'Cho biết các đặc điểm chính của chủ nghĩ…' (58 chars) → 'chủ nghĩa xã hội khoa học lý luận học th…' (122 chars)
2026-04-15 13:41:48,356 - knowgen.retriever - INFO - Searching 20 candidates…
2026-04-15 13:41:48,504 - knowgen.retriever - INFO - Found 20 candidates
2026-04-15 13:41:48,504 - knowgen.retriever - INFO - Doc-relevance gate: 1/1 doc(s) passed, 20/20 chunks retained.
2026-04-15 13:41:48,686 - knowgen.retriever - INFO - After ranking & filtering: 3 documents
2026-04-15 13:41:48,687 - knowgen.retriever - INFO - Retrieval complete: 3 docs, 3 evidence items
2026-04-15 13:41:48,687 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


...

✓ Rewritten Query: NNHT.pdf tài liệu NNHT nguyên lý học tập nguyên tắc học tập lý thuyết học tập phương pháp học tập cơ chế học tập định nghĩa khái niệm đặc điểm vai trò ứng dụng ý nghĩa tóm tắt tổng quan nội dung chính thông tin cốt lõi các nguyên lý cơ bản chung
✓ Documents Retrieved: 1

📄 Top Documents:
  [1] Score: 0.830 | Header: **1. biểu thức chính quy**
      Preview: passage: Summary of NNHT.pdf:
Tài liệu là giáo trình "Ngôn ngữ hình thức & ôtômá...

💡 Evidence Summaries (1 items):
  1. passage: Summary of NNHT

[Query 3] Định lý Pythagore phát biểu như thế nào và có ứng dụng gì trong hình học?


2026-04-15 13:41:58,588 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 429 Too Many Requests"
2026-04-15 13:41:58,588 - google_genai._api_client - INFO - Retrying google.genai._api_client.BaseApiClient._request_once in 1.6 seconds as it raised ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 2.401547167s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev

✓ Rewritten Query: Định lý Pythagore Định lí Pytago Pitago phát biểu nội dung công thức định nghĩa mệnh đề, ứng dụng thực tế vai trò tầm quan trọng áp dụng trong hình học phẳng hình học không gian tam giác vuông cạnh huyền cạnh góc vuông, ví dụ giải bài toán toán học kiến trúc kỹ thuật
✓ Documents Retrieved: 0
⚠️  No documents retrieved for this query

[Query 4] Tạo một câu hỏi trắc nghiệm về giao thoa ánh sáng và các điều kiện cần thiết.


2026-04-15 13:42:11,174 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-15 13:42:11,197 - knowgen.retriever - INFO - Query rewritten: 'Tạo một câu hỏi trắc nghiệm về giao thoa…' (77 chars) → 'giao thoa ánh sáng, điều kiện cần thiết …' (390 chars)
2026-04-15 13:42:11,206 - knowgen.retriever - INFO - Searching 20 candidates…
2026-04-15 13:42:11,342 - knowgen.retriever - INFO - Found 20 candidates
2026-04-15 13:42:11,342 - knowgen.retriever - INFO - Doc-relevance gate: 1/1 doc(s) passed, 20/20 chunks retained.
2026-04-15 13:42:11,544 - knowgen.retriever - INFO - After ranking & filtering: 5 documents
2026-04-15 13:42:11,545 - knowgen.retriever - INFO - Retrieval complete: 5 docs, 5 evidence items


✓ Rewritten Query: giao thoa ánh sáng, điều kiện cần thiết cho giao thoa ánh sáng, hiện tượng giao thoa ánh sáng, nguyên lý giao thoa, sóng ánh sáng, nguồn kết hợp, thí nghiệm Young, vân sáng, vân tối, bước sóng, hiệu đường đi, pha, tần số, độ lệch pha, ứng dụng giao thoa, khái niệm giao thoa, tính chất sóng ánh sáng, phân biệt giao thoa nhiễu xạ, điều kiện để quan sát giao thoa rõ nét, công thức giao thoa
✓ Documents Retrieved: 5

📄 Top Documents:
  [1] Score: 0.848 | Header: **10.1. giới thiệu vềgiao thoa 2. hiện tượng giao thoa qua hai khe hẹp (thí nghiệm young)** _**b) điều kiện để có cực đại, cực tiểu giao thoa:**_
      Preview: passage: Summary of Chương 10 - Giao thoa ánh sáng.pdf:
Chương 10 giới thiệu về...
  [2] Score: 0.845 | Header: **vật lí 1 cơ học, nhiệt động lực học và quang học** > **10.1. giới thiệu vềgiao thoa**
      Preview: passage: Summary of Chương 10 - Giao thoa ánh sáng.pdf:
Chương 10 giới thiệu về...
  [3] Score: 0.843 | Header: **10.1. giới thiệu vềgiao thoa

## Section 7: Compare & Analyze Results

In [11]:
print("=" * 80)
print("📊 COMPARATIVE ANALYSIS")
print("=" * 80)

# Aggregate statistics
analysis_results = {
    "total_queries": len(test_queries),
    "supervisor_success": 0,
    "retriever_success": 0,
    "task_type_distribution": {},
    "language_distribution": {},
    "avg_docs_retrieved": 0,
    "total_docs_retrieved": 0,
    "avg_similarity_scores": []
}

print("\n1️⃣  SUPERVISOR OUTPUT SUMMARY:")
print("-" * 80)
for query_idx, (query_key, sup_result) in enumerate(supervisor_results.items(), 1):
    if "error" not in sup_result:
        analysis_results["supervisor_success"] += 1
        task_type = sup_result.get("task_type", "unknown")
        language = sup_result.get("language", "unknown")
        
        # Update distributions
        analysis_results["task_type_distribution"][task_type] = \
            analysis_results["task_type_distribution"].get(task_type, 0) + 1
        analysis_results["language_distribution"][language] = \
            analysis_results["language_distribution"].get(language, 0) + 1
        
        intent = sup_result.get('intent_summary', 'N/A')[:50]
        print(f"[Query {query_idx}] {task_type.upper():7} | Language: {language:6} | {intent}")
    else:
        print(f"[Query {query_idx}] ERROR - {sup_result.get('error', 'Unknown error')[:50]}")

if len(retriever_results) == 0:
    print("\n2️⃣  RETRIEVER OUTPUT SUMMARY:")
    print("-" * 80)
    print("⚠️  No retriever results available")
    print("   (FAISS index may not be created yet)")
else:
    print("\n2️⃣  RETRIEVER OUTPUT SUMMARY:")
    print("-" * 80)
    for query_idx, (query_key, ret_result) in enumerate(retriever_results.items(), 1):
        if "error" not in ret_result:
            analysis_results["retriever_success"] += 1
            docs_count = len(ret_result.get("retrieved_docs", []))
            analysis_results["total_docs_retrieved"] += docs_count
            
            # Collect similarity scores
            for doc in ret_result.get("retrieved_docs", []):
                metadata = doc.metadata if hasattr(doc, 'metadata') else {}
                score = metadata.get('similarity_score', 0) if metadata else 0
                if score > 0:
                    analysis_results["avg_similarity_scores"].append(score)
            
            rewritten = ret_result.get('rewritten_query', '')[:50]
            print(f"[Query {query_idx}] Retrieved: {docs_count:2} docs | Rewritten: {rewritten}")
        else:
            print(f"[Query {query_idx}] ERROR - {ret_result.get('error', 'Unknown error')[:50]}")

if analysis_results["retriever_success"] > 0:
    analysis_results["avg_docs_retrieved"] = \
        analysis_results["total_docs_retrieved"] / analysis_results["retriever_success"]

print("\n3️⃣  STATISTICS SUMMARY:")
print("-" * 80)
print(f"✓ Total Queries: {analysis_results['total_queries']}")
print(f"✓ Supervisor Success: {analysis_results['supervisor_success']}/{analysis_results['total_queries']} ({(analysis_results['supervisor_success']/analysis_results['total_queries']*100):.0f}%)")

if analysis_results['retriever_success'] > 0 or len(retriever_results) > 0:
    retriever_rate = (analysis_results['retriever_success']/analysis_results['total_queries']*100) if analysis_results['total_queries'] > 0 else 0
    print(f"✓ Retriever Success: {analysis_results['retriever_success']}/{analysis_results['total_queries']} ({retriever_rate:.0f}%)")
    print(f"✓ Average Docs per Query: {analysis_results['avg_docs_retrieved']:.1f}")
    print(f"✓ Total Documents Retrieved: {analysis_results['total_docs_retrieved']}")
else:
    print(f"⚠️  Retriever tests not available (FAISS index not initialized)")

if analysis_results["avg_similarity_scores"]:
    avg_score = sum(analysis_results["avg_similarity_scores"]) / len(analysis_results["avg_similarity_scores"])
    min_score = min(analysis_results["avg_similarity_scores"])
    max_score = max(analysis_results["avg_similarity_scores"])
    print(f"✓ Similarity Scores: Avg={avg_score:.3f}, Min={min_score:.3f}, Max={max_score:.3f}")

if analysis_results["task_type_distribution"]:
    print(f"\n✓ Task Type Distribution: {analysis_results['task_type_distribution']}")
if analysis_results["language_distribution"]:
    print(f"✓ Language Distribution: {analysis_results['language_distribution']}")

print("\n" + "=" * 80)

📊 COMPARATIVE ANALYSIS

1️⃣  SUPERVISOR OUTPUT SUMMARY:
--------------------------------------------------------------------------------
[Query 1] QA      | Language: vi     | Người dùng muốn biết các đặc điểm chính của chủ ng
[Query 2] QA      | Language: vi     | Người dùng muốn tóm tắt các nguyên lý học tập được
[Query 3] QA      | Language: vi     | Người dùng muốn biết định lý Pythagore phát biểu n
[Query 4] QUIZ    | Language: vi     | Người dùng muốn tạo một câu hỏi trắc nghiệm về gia

2️⃣  RETRIEVER OUTPUT SUMMARY:
--------------------------------------------------------------------------------
[Query 1] Retrieved:  3 docs | Rewritten: chủ nghĩa xã hội khoa học lý luận học thuyết quan 
[Query 2] Retrieved:  1 docs | Rewritten: NNHT.pdf tài liệu NNHT nguyên lý học tập nguyên tắ
[Query 3] Retrieved:  0 docs | Rewritten: Định lý Pythagore Định lí Pytago Pitago phát biểu 
[Query 4] Retrieved:  5 docs | Rewritten: giao thoa ánh sáng, điều kiện cần thiết cho giao t

3️⃣  STATISTICS S